# フェーズ1：データ確定（EdNet 監査サンプル）

**目的**：本番評価の前に、①使う KT 階層とカラム、②4カテゴリの定義、③決定点の定義、④セッション閾値 を確定する。

**このノートブックの対象は監査サンプル（32人）のみ。** ここでの数値は設計確認用であり、研究結果として報告しない。

## 進め方
1. 「設定」セルの `DATA_DIR` を自分の環境に合わせて書き換える
2. 上から順に実行する
3. 各セクション末尾の **【記録】** 欄に、確認した事実と決定事項を書き込む（事実と判断を分けて書く）

## フェーズ1 ToDo
- [ ] 1. KT階層（KT3 / KT4）とカラムを確定
- [ ] 2. 4カテゴリの対応表を文書化
- [ ] 3. UI構造で決まる遷移の割合を集計 → 決定点を定義
- [ ] 4. 60分セッション閾値の妥当性を確認
- [ ] 5. 前処理パイプラインを監査サンプルで検証

## 0. 設定
EdNet KT3/KT4 は学習者ごとに `u{user_id}.csv` が1ファイルずつある構成（Choi et al., AIED 2020）。
ダウンロード・解凍したフォルダを `DATA_DIR` に指定する。Google Colab の場合は Drive をマウントしてそのパスを書く。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ===== ここを書き換える =====
DATA_DIR = Path("data/audit/KT3")   # u*.csv が入っているフォルダ
KT_LEVEL = "KT3"                     # "KT3" or "KT4"
N_AUDIT = 32                         # 監査サンプル人数
SEED = 0                             # サンプリングの乱数シード（記録しておく）
SESSION_GAP_MIN = 60                 # セッション区切り閾値（分）
# ===========================

rng = np.random.default_rng(SEED)
files = sorted(DATA_DIR.glob("u*.csv"))
print(f"学習者ファイル数: {len(files)}")
assert len(files) > 0, "DATA_DIR に u*.csv が見つからない。パスを確認する。"

学習者ファイル数: 32


## 1. 読み込みとカラム確認
カラム名はドキュメントの記憶に頼らず、**実ファイルで確認する**。

In [2]:
audit_files = rng.choice(files, size=min(N_AUDIT, len(files)), replace=False)

dfs = []
for f in audit_files:
    d = pd.read_csv(f)
    d.insert(0, "user_id", f.stem)
    dfs.append(d)
raw = pd.concat(dfs, ignore_index=True)

print("カラム:", list(raw.columns))
print(raw.dtypes)
print(f"\n行数: {len(raw):,} / 学習者数: {raw.user_id.nunique()}")
print("\n欠損率:\n", raw.isna().mean().round(3))
raw.head(20)

カラム: ['user_id', 'timestamp', 'action_type', 'item_id', 'source', 'user_answer', 'platform']
user_id          str
timestamp      int64
action_type      str
item_id          str
source           str
user_answer      str
platform         str
dtype: object

行数: 14,102 / 学習者数: 32

欠損率:
 user_id        0.000
timestamp      0.000
action_type    0.000
item_id        0.000
source         0.000
user_answer    0.731
platform       0.000
dtype: float64


,user_id,timestamp,action_type,item_id,source,user_answer,platform
0,u557164,1554184149351,enter,b5766,diagnosis,NaN,mobile
1,u557164,1554184204059,respond,q8415,diagnosis,c,mobile
2,u557164,1554184209075,submit,b5766,diagnosis,NaN,mobile
3,u557164,1554184209144,enter,b5651,diagnosis,NaN,mobile
4,u557164,1554184231265,respond,q8180,diagnosis,d,mobile
5,u557164,1554184235777,submit,b5651,diagnosis,NaN,mobile
6,u557164,1554184235860,enter,b377,diagnosis,NaN,mobile
7,u557164,1554184252420,respond,q377,diagnosis,a,mobile
8,u557164,1554184257034,submit,b377,diagnosis,NaN,mobile
9,u557164,1554184257098,enter,b5166,diagnosis,NaN,mobile


In [3]:
# 時刻の単位を確認（EdNet は Unix ミリ秒のはずだが、桁数で確かめる）
ts = raw["timestamp"]
print("timestamp の桁数:", ts.astype(str).str.len().value_counts().to_dict())
raw["time"] = pd.to_datetime(ts, unit="ms")
print("期間:", raw["time"].min(), "〜", raw["time"].max())

# 学習者内で時刻が単調増加か（同時刻の行がどれだけあるか）
raw = raw.sort_values(["user_id", "timestamp"], kind="stable").reset_index(drop=True)
dup = raw.duplicated(["user_id", "timestamp"], keep=False).mean()
print(f"同一タイムスタンプを持つ行の割合: {dup:.3f}  ← 高い場合、行の順序の扱いを決める必要がある")

timestamp の桁数: {13: 14102}
期間: 2018-10-07 10:15:09.997000 〜 2019-11-26 09:17:54.095000
同一タイムスタンプを持つ行の割合: 0.001  ← 高い場合、行の順序の扱いを決める必要がある


In [4]:
# action_type と item の種類（item_id の先頭文字: q=問題, b=バンドル, e=解説, l=講義 を想定。実データで確認する）
raw["item_type"] = raw["item_id"].astype(str).str[0]
print("action_type:\n", raw["action_type"].value_counts(), "\n")
print("item_type:\n", raw["item_type"].value_counts(), "\n")
if "source" in raw.columns:
    print("source:\n", raw["source"].value_counts(), "\n")
if "platform" in raw.columns:
    print("platform:\n", raw["platform"].value_counts())

pd.crosstab(raw["action_type"], raw["item_type"])

action_type:
 action_type
enter      5157
respond    3788
submit     2678
quit       2479
Name: count, dtype: int64 

item_type:
 item_type
b    5356
e    4936
q    3788
l      22
Name: count, dtype: int64 

source:
 source
sprint            13095
diagnosis           708
adaptive_offer      182
review_quiz          65
my_note              28
tutor                12
review               12
Name: count, dtype: int64 

platform:
 platform
mobile    13130
web         972
Name: count, dtype: int64


item_type,b,e,l,q
action_type,,,,
enter,2678,2468,11,0
quit,0,2468,11,0
respond,0,0,0,3788
submit,2678,0,0,0


**【記録】1. KT階層とカラム**
- 事実：（実際のカラム名、timestamp の単位、同時刻行の割合 など）
- 決定：（KT3 / KT4 のどちらを使うか、その理由）

## 2. 4カテゴリの対応表
`(action_type, item_type)` の組をカテゴリに割り当てる。**下の `CATEGORY_MAP` は記入例であり、自分の定義に書き換える。**
未割り当ての組は自動で一覧表示されるので、すべて埋まるまで修正する（対象外にする組は `None` を明示的に入れる）。

In [ ]:
# ===== 記入例：自分の4カテゴリ定義に置き換える =====
CATEGORY_MAP = {
    ("enter",  "b"): "solve",
    ("respond","q"): "solve",
    ("submit", "b"): "solve",
    ("enter",  "e"): "review",
    ("quit",   "e"): "review",
    ("enter",  "l"): "lecture",
    ("quit",   "l"): "lecture",
    # ("play_audio", "b"): "aux",
    # ("pay", ...): None,   # 分析対象外は None
}
# ====================================================

pairs = raw[["action_type", "item_type"]].drop_duplicates()
unmapped = pairs[[ (a, i) not in CATEGORY_MAP for a, i in pairs.itertuples(index=False) ]]
cnt = raw.groupby(["action_type", "item_type"]).size().rename("n")
print(f"未割り当ての組: {len(unmapped)} 種類")
unmapped.merge(cnt.reset_index(), on=["action_type", "item_type"]).sort_values("n", ascending=False)

In [ ]:
raw["category"] = [CATEGORY_MAP.get((a, i), "UNMAPPED") for a, i in zip(raw.action_type, raw.item_type)]
excluded = raw["category"].isna()
print(f"対象外 (None) にした行の割合: {excluded.mean():.3f}")
print(f"未割り当ての行の割合: {(raw.category == 'UNMAPPED').mean():.3f}  ← 0 になるまで修正する")
print(raw.loc[~excluded, "category"].value_counts(normalize=True).round(3))

**【記録】2. カテゴリ定義**
- 各カテゴリの意味（1行ずつ）と、どの `(action_type, item_type)` を含むか
- 対象外にした行動と、その理由
- 4カテゴリの頻度の偏り（極端に少ないカテゴリがあれば Macro-F1 の解釈に注意）

## 3. UI構造で決まる遷移の割合 → 決定点の定義
細かい行動トークン `(action_type, item_type)` の一次遷移を数え、**直前の状態から次がほぼ確定する遷移**がどれくらいあるかを見る。
ここが大きいほど、全イベントでの予測精度は「アプリの画面フロー」を当てているだけになる。

In [ ]:
DET_THRESHOLD = 0.95   # 次の行動の最大確率がこれ以上なら「ほぼ確定」とみなす

ev = raw[raw["category"].notna() & (raw["category"] != "UNMAPPED")].copy()
ev["token"] = ev["action_type"] + ":" + ev["item_type"]
ev["next_token"] = ev.groupby("user_id")["token"].shift(-1)
tr = ev.dropna(subset=["next_token"])

P = pd.crosstab(tr["token"], tr["next_token"], normalize="index")
stats = pd.DataFrame({
    "n": tr["token"].value_counts(),
    "max_p": P.max(axis=1),
    "argmax_next": P.idxmax(axis=1),
    "entropy_bits": -(P * np.log2(P.where(P > 0))).sum(axis=1),
}).sort_values("n", ascending=False)
stats["near_deterministic"] = stats["max_p"] >= DET_THRESHOLD

share = stats.loc[stats.near_deterministic, "n"].sum() / stats["n"].sum()
print(f"ほぼ確定 (max_p >= {DET_THRESHOLD}) な状態から出る遷移の割合: {share:.3f}")
stats.round(3)

In [ ]:
# カテゴリ単位でも同じことを確認（4クラスに集約した後の構造的決定性）
ev["next_category"] = ev.groupby("user_id")["category"].shift(-1)
Pc = pd.crosstab(ev["category"], ev["next_category"], normalize="index").round(3)
print("カテゴリ遷移行列（行=直前、列=次）")
Pc

In [ ]:
# 決定点候補：『ほぼ確定でない』状態の直後。まずは submit 後に何を選ぶかを見る
cand = stats.index[~stats.near_deterministic].tolist()
print("決定点候補となる直前トークン:", cand)

after_submit = tr[tr["token"].str.startswith("submit")]
print("\nsubmit の直後の行動分布:")
print(after_submit["next_token"].value_counts(normalize=True).round(3).head(15))

**【記録】3. 決定点**
- 事実：ほぼ確定な遷移の割合、決定点候補のトークン
- 決定：本評価で使う決定点の定義（例：「submit の直後、および解説/講義を閉じた直後」）
- 注意：決定点の定義は本番データを見て変えないこと（評価データを見ながら定義を調整すると楽観的なバイアスが入る）

## 4. セッション閾値（60分）の妥当性
同一学習者内の行動間隔の分布を対数軸で描き、閾値が「谷」に位置するか確認する。あわせて、閾値を変えたときのセッション数の変化（感度）を見る。

In [ ]:
raw["gap_min"] = raw.groupby("user_id")["timestamp"].diff() / 60000
gaps = raw["gap_min"].dropna()
gaps_pos = gaps[gaps > 0]

fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.logspace(np.log10(gaps_pos.min()), np.log10(gaps_pos.max()), 80)
ax.hist(gaps_pos, bins=bins, color="#4C72B0")
ax.set_xscale("log")
for t in [10, 30, 60, 120]:
    ax.axvline(t, ls="--", lw=1, color="gray")
    ax.text(t, ax.get_ylim()[1] * 0.9, f"{t}m", ha="center", fontsize=8)
ax.set_xlabel("inter-event gap [min] (log scale)"); ax.set_ylabel("count")  # 日本語フォント無しでも文字化けしないよう英語表記
ax.set_title("Inter-event gaps within learner")
plt.tight_layout(); plt.show()

print(f"間隔0の行の割合: {(gaps == 0).mean():.3f}")

In [ ]:
rows = []
for th in [10, 20, 30, 60, 90, 120, 240]:
    new_sess = raw["gap_min"].isna() | (raw["gap_min"] > th)
    sid = new_sess.groupby(raw["user_id"]).cumsum()
    sizes = raw.groupby([raw["user_id"], sid]).size()
    rows.append({"閾値(分)": th, "セッション数": len(sizes),
                 "セッション長の中央値(行動数)": sizes.median(),
                 "1行動だけのセッション割合": (sizes == 1).mean()})
pd.DataFrame(rows)

**【記録】4. セッション閾値**
- 事実：分布の谷の位置、閾値を変えたときのセッション数の変化
- 決定：60分を採用するか。採用する場合も、感度分析（例：30分・120分）を本番で報告するか

## 5. 前処理パイプラインの確定
上で決めた定義を1つの関数にまとめる。本番データにも**このまま**適用する。

In [ ]:
def preprocess(df_raw: pd.DataFrame, category_map: dict, session_gap_min: float) -> pd.DataFrame:
    d = df_raw.sort_values(["user_id", "timestamp"], kind="stable").copy()
    d["item_type"] = d["item_id"].astype(str).str[0]
    d["category"] = [category_map.get((a, i), "UNMAPPED") for a, i in zip(d.action_type, d.item_type)]
    if (d["category"] == "UNMAPPED").any():
        raise ValueError("未割り当ての (action_type, item_type) がある")
    d = d[d["category"].notna()]
    gap = d.groupby("user_id")["timestamp"].diff() / 60000
    d["session_id"] = (gap.isna() | (gap > session_gap_min)).groupby(d["user_id"]).cumsum()
    d["pos_in_session"] = d.groupby(["user_id", "session_id"]).cumcount()
    d["next_category"] = d.groupby(["user_id", "session_id"])["category"].shift(-1)  # セッション末尾は NaN
    return d.reset_index(drop=True)

data = preprocess(raw.drop(columns=["category"]), CATEGORY_MAP, SESSION_GAP_MIN)

# 監査チェック
print("学習者数:", data.user_id.nunique(), "/ 行数:", len(data))
print("学習者あたり行動数:", data.groupby("user_id").size().describe().round(1).to_dict())
print("予測対象（次カテゴリあり）の行数:", data["next_category"].notna().sum())
assert (data.groupby("user_id")["timestamp"].diff().dropna() >= 0).all(), "時刻の逆転がある"
data.head(15)

**【記録】5. パイプライン監査**
- 手作業で1〜2人分のログを目視し、カテゴリ・セッション区切りが意図通りか確認したか
- セッション末尾の扱い（「セッション終了」を5つ目のクラスにするか、予測対象から外すか）← 要決定。終了を予測しないと、シミュレーションでセッション長を生成できない